# Logical and Physical Plans: Deep Reading and Plan Forensics

**Module**: 07_spark_sql_internals  
**Notebook**: 02_logical_and_physical_plans.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Step-by-step guide to reading explain() outputs: Project, Filter, Join, Aggregate, Exchange, and Sort nodes.
- **Why does this topic exist?**: Reading execution plans is the primary method to detect bottlenecks before running long jobs.
- **What problem does it solve?**: Blindly running queries without knowing if they will shuffle or broadcast.
- **Where is it used in real-world Data Engineering?**: Query tuning in production environments.

---

## 2. Reading Plan Operators

- `FileScan`: Table reader (look for PushedFilters and PartitionFilters).
- `Project`: Column selection / expression calculation.
- `Filter`: Row filtering.
- `Exchange`: Shuffle boundary.
- `HashAggregate` / `SortAggregate`: Grouping operation.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Plans_Forensics_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
p = orders.filter("status = 'Completed'").groupBy("channel").count().orderBy("count")
p.explain("formatted")

---

## Experiment: Extended Plan Inspection

- **1. What are we testing?**: Print explain('extended') to compare analyzed vs optimized plans.
- **2. What do we expect?**: Optimized plan is noticeably simpler.

In [ ]:
p.explain('extended')

**3. What happened?**: Unneeded projections and expressions were collapsed.  
**4. Why did it happen?**: Catalyst removed redundant nodes.  
**5. Practical Engineering Lesson**: Optimized plans show the true operations that will run.

---

## Practical Exercises

### Exercise 1 (Beginner): Find Root Operator
**Problem Statement**: Identify the top-level operator in a query plan.  
**Input / Context**: p

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Count Projects
**Problem Statement**: Count Project operators in plan.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Cost Plan
**Problem Statement**: Run explain('cost') and check statistics.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Find Root Operator):
```python
print('Top operator is SortExec or AdaptiveSparkPlan')
```

#### Solution 2 (Count Projects):
```python
print('Inspected plan strings')
```

#### Solution 3 (Cost Plan):
```python
p.explain('cost')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Plan Reading**: Scan -> Filter -> Project -> Exchange -> Aggregate -> Sort.

### Essential Technical Terms
- `ExecutedPlan`: Physical plan after code generation.
- `AST`: Tree of relational operators.

### Common Engineering Mistakes
- **Confusing Parsed with Optimized**: Parsed plan still contains unverified syntax.

### Interview Questions on This Topic
- **Q: How do you view all 4 plan stages?**
  - *A*: Call df.explain(True) or df.explain('extended').

### What to Study Next
Open 03_tungsten_and_whole_stage_codegen.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')